# RSNA Knee Abnormality Detection — V3 MedSigLIP Fast Pipeline

Competition-oriented redesign for an offline Kaggle notebook with two T4 GPUs.

Pipeline:
**CSV/series metadata → metadata-first slot selection → streaming DICOM slices → MedSigLIP embeddings → study pooling → 12-label ranking heads → optional precomputed text-score fusion → submission**

Runtime principles:
- Never recursively scan the full `/kaggle/input` tree.
- Never construct the 14.8 GB pixel cache seen in the previous run.
- Never load Gemma in the critical image pipeline.
- Persist embeddings, not pixels.
- Use both T4s for independent image-embedding batches.
- Keep report/Gemma scores as an optional separate artifact.
- Stop expensive optional stages when the runtime guard is reached.

This notebook does not guarantee a leaderboard score.

In [ ]:
# 1. FAST OFFLINE ENVIRONMENT
import os
os.environ["PYDEVD_DISABLE_FILE_VALIDATION"] = "1"
os.environ["PYTHONWARNINGS"] = "ignore"
os.environ["OMP_NUM_THREADS"] = "4"
os.environ["MKL_NUM_THREADS"] = "4"
os.environ["OPENBLAS_NUM_THREADS"] = "4"
os.environ["NUMEXPR_NUM_THREADS"] = "4"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import gc, re, time, math, random, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    torch.set_num_threads(4)
    torch.set_num_interop_threads(1)
except Exception:
    pass

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

START_TIME = time.time()
MAX_RUNTIME_HOURS = float(os.environ.get("MAX_RUNTIME_HOURS", "3.5"))

def elapsed():
    return time.time() - START_TIME

def hours_left():
    return MAX_RUNTIME_HOURS - elapsed()/3600.0

def log(x):
    print(f"[{elapsed():8.1f}s] {x}", flush=True)

DEVICES = [torch.device(f"cuda:{i}") for i in range(torch.cuda.device_count())]
if not DEVICES:
    DEVICES = [torch.device("cpu")]

log(f"devices={DEVICES}")
for d in DEVICES:
    if d.type == "cuda":
        log(f"{d}: {torch.cuda.get_device_name(d.index)}")
log(f"runtime budget={MAX_RUNTIME_HOURS:.2f} h")

## 2. Configuration

Set `DATASET_DIR` and `MEDSIGLIP_DIR` explicitly if automatic shallow discovery does not match your Kaggle attachment layout.

Do not replace this with `rglob("*")`.

In [ ]:
# 2. CONFIG
INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working/rsna_knee_v3")
WORK_ROOT.mkdir(parents=True, exist_ok=True)

# Kaggle attachment name can be correct while the CSVs live one or more
# directories below it. We therefore discover ONLY the small metadata files,
# not the 500 GB image tree.
DATASET_DIR = INPUT_ROOT / "rsna-knee-abnormality-detection"

TRAIN_CSV = None
TEST_CSV = None
TRAIN_SERIES_CSV = None
TEST_SERIES_CSV = None
SAMPLE_SUB = None

# Explicit override is preferable if the attachment layout is known.
MEDSIGLIP_DIR = None

TARGETS = [
    "ACL", "MCL", "Medial_Meniscus", "Lateral_Meniscus",
    "Medial_OA", "Lateral_OA", "PF_OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture"
]

MAX_SLOTS = 6
SLICES_PER_SLOT = 2
IMAGE_SIZE = 448
EMBED_BATCH = 16

TEXT_TRAIN = WORK_ROOT / "train_text_logits.csv"
TEXT_TEST = WORK_ROOT / "test_text_logits.csv"

def find_metadata_files(root, max_depth=4):
    """
    Find only CSV metadata files within a small directory depth.
    This deliberately does NOT scan the DICOM tree recursively.
    """
    root = Path(root)
    found = {}
    if not root.exists():
        return found

    queue = [(root, 0)]
    while queue:
        d, depth = queue.pop()
        try:
            entries = list(d.iterdir())
        except (PermissionError, OSError):
            continue

        for p in entries:
            if p.is_file() and p.suffix.lower() == ".csv":
                name = p.name.lower()
                found.setdefault(name, p)

            elif p.is_dir() and depth < max_depth:
                # Do not descend into obvious image/data directories.
                n = p.name.lower()
                if n not in {
                    "train_images", "test_images", "images", "dicom",
                    "train", "test", "series", "studies"
                }:
                    queue.append((p, depth + 1))
    return found

def pick_csv(files, exact_names):
    for name in exact_names:
        if name.lower() in files:
            return files[name.lower()]
    return None

metadata_files = find_metadata_files(DATASET_DIR, max_depth=4)

# If the named attachment directory itself does not exist, inspect only the
# first level of /kaggle/input for another matching dataset directory.
if not metadata_files:
    candidates = []
    try:
        candidates = [p for p in INPUT_ROOT.iterdir() if p.is_dir()]
    except Exception:
        pass

    for d in candidates:
        files = find_metadata_files(d, max_depth=3)
        if "train.csv" in files or "test.csv" in files:
            DATASET_DIR = d
            metadata_files = files
            break

TRAIN_CSV = pick_csv(metadata_files, ["train.csv"])
TEST_CSV = pick_csv(metadata_files, ["test.csv"])
TRAIN_SERIES_CSV = pick_csv(metadata_files, [
    "train_series.csv", "train_series_metadata.csv"
])
TEST_SERIES_CSV = pick_csv(metadata_files, [
    "test_series.csv", "test_series_metadata.csv"
])
SAMPLE_SUB = pick_csv(metadata_files, [
    "sample_submission.csv", "sample_submission.csv.gz"
])

log(f"dataset root={DATASET_DIR}")
log(f"metadata files discovered={len(metadata_files)}")
log(f"train={TRAIN_CSV}")
log(f"test={TEST_CSV}")
log(f"train_series={TRAIN_SERIES_CSV}")
log(f"test_series={TEST_SERIES_CSV}")
log(f"sample_submission={SAMPLE_SUB}")

if TRAIN_CSV is None or TEST_CSV is None:
    # Print only small metadata inventory to make the failure actionable.
    visible = sorted(str(p) for p in metadata_files.values())
    raise FileNotFoundError(
        "Could not locate train.csv and test.csv within the metadata search "
        f"depth. DATASET_DIR={DATASET_DIR}. Found CSVs:\n" +
        "\n".join(visible[:100])
    )

In [ ]:
# 3. LOAD CSV METADATA ONLY
def require_csv(path, label):
    if not path.exists():
        raise FileNotFoundError(f"{label} not found: {path}")
    return pd.read_csv(path)

train = require_csv(TRAIN_CSV, "train.csv")
test = require_csv(TEST_CSV, "test.csv")
train_series = pd.read_csv(TRAIN_SERIES_CSV) if TRAIN_SERIES_CSV.exists() else None
test_series = pd.read_csv(TEST_SERIES_CSV) if TEST_SERIES_CSV.exists() else None

def norm_name(x):
    return re.sub(r"[^a-z0-9]+", "_", str(x).lower()).strip("_")

def find_col(df, names):
    lookup = {norm_name(c): c for c in df.columns}
    for n in names:
        if norm_name(n) in lookup:
            return lookup[norm_name(n)]
    return None

ID_COL = find_col(train, ["StudyInstanceUID", "study_id", "id"])
if ID_COL is None:
    raise ValueError(f"Study ID column not found: {list(train.columns)}")

target_map = {}
lookup = {norm_name(c): c for c in train.columns}
for t in TARGETS:
    if norm_name(t) in lookup:
        target_map[t] = lookup[norm_name(t)]

log(f"train={train.shape}, test={test.shape}")
log(f"train_series={None if train_series is None else train_series.shape}")
log(f"resolved targets={target_map}")

In [ ]:
# 3A. METADATA SANITY CHECK
print("train columns:", list(train.columns))
print("test columns:", list(test.columns))
if train_series is not None:
    print("train_series columns:", list(train_series.columns))
    print(train_series.head(3).to_string(index=False))
if test_series is not None:
    print("test_series columns:", list(test_series.columns))
    print(test_series.head(3).to_string(index=False))

## 4. Metadata-first series selection

The competition metadata provides `Anatomical_Plane`, `Fluid_Sensitive`, and `Fat_Suppression`. We rank series from these fields and choose a small, diverse set per study.

No DICOM headers are opened during selection.

In [ ]:
# 4. SERIES SELECTION FROM CSV METADATA
def canonical_series(df):
    if df is None:
        return None
    x = df.copy()
    lookup = {norm_name(c): c for c in x.columns}
    aliases = {
        "study": ["StudyInstanceUID", "study_id", "study"],
        "series": ["SeriesInstanceUID", "series_id", "series"],
        "plane": ["Anatomical_Plane", "plane"],
        "fluid": ["Fluid_Sensitive"],
        "fat": ["Fat_Suppression"],
    }
    for dst, names in aliases.items():
        for n in names:
            if norm_name(n) in lookup:
                x[dst] = x[lookup[norm_name(n)]]
                break
    return x

def value_true(x):
    return str(x).strip().lower() in {"1","true","yes","y","fluid sensitive","fat suppressed"}

def series_score(r):
    plane = str(r.get("plane","")).lower()
    s = 0.0
    if "sag" in plane: s += 3.0
    elif "cor" in plane: s += 2.5
    elif "ax" in plane: s += 2.0
    if value_true(r.get("fluid","")): s += 3.0
    if value_true(r.get("fat","")): s += 1.5
    return s

def select_slots(df, max_slots=6):
    if df is None:
        return {}
    x = canonical_series(df)
    if not {"study","series"}.issubset(x.columns):
        raise ValueError(f"Series CSV lacks study/series identifiers: {list(x.columns)}")
    x["_score"] = x.apply(series_score, axis=1)
    result = {}
    for study, g in x.groupby("study", sort=False):
        g = g.sort_values("_score", ascending=False)
        chosen = []
        for pk in ("sag","cor","ax"):
            q = g[g["plane"].astype(str).str.lower().str.contains(pk, na=False)]
            if len(q):
                chosen.append(q.iloc[0])
        for _, r in g.iterrows():
            if len(chosen) >= max_slots:
                break
            if all(str(r["series"]) != str(z["series"]) for z in chosen):
                chosen.append(r)
        result[str(study)] = [dict(r) for r in chosen[:max_slots]]
    return result

train_slots = select_slots(train_series, MAX_SLOTS)
test_slots = select_slots(test_series, MAX_SLOTS)

log(f"selected train studies={len(train_slots)}")
log(f"selected test studies={len(test_slots)}")

## 5. Resolve only selected series

Kaggle layouts vary. The resolver checks common explicit paths. If your DICOM root is different, modify only `SERIES_ROOTS` or `resolve_series_dir()`.

It never scans all DICOM files just to discover metadata.

In [ ]:
# 5. SERIES DIRECTORY RESOLUTION
#
# Important Kaggle detail:
# the CSV metadata can be mounted under /kaggle/input/competitions while
# the actual DICOM payload may be in a sibling attachment/root. Therefore
# image roots are discovered independently of the CSV root.

def immediate_dirs(root):
    try:
        return [p for p in Path(root).iterdir() if p.is_dir()]
    except Exception:
        return []

# Start with the dataset root, then every first-level Kaggle attachment.
IMAGE_ROOTS = [DATASET_DIR]
for p in immediate_dirs(INPUT_ROOT):
    if p not in IMAGE_ROOTS:
        IMAGE_ROOTS.append(p)

# Also include common nested roots without recursively scanning them.
expanded=[]
for r in IMAGE_ROOTS:
    expanded.append(r)
    for name in ("train", "test", "train_images", "test_images",
                 "images", "dicom", "data"):
        p=r/name
        if p.is_dir():
            expanded.append(p)
IMAGE_ROOTS=expanded

log("candidate image roots:")
for r in IMAGE_ROOTS:
    log(f"  {r}")

def direct_candidates(root, study, series):
    study, series = str(study), str(series)
    return [
        root/study/series,
        root/series,
        root/"train"/study/series,
        root/"test"/study/series,
        root/"train_images"/study/series,
        root/"test_images"/study/series,
        root/"images"/study/series,
        root/"dicom"/study/series,
        root/"data"/study/series,
    ]

def resolve_series_dir(study, series):
    # Fast path. This handles the normal Kaggle layouts.
    for root in IMAGE_ROOTS:
        for p in direct_candidates(root, study, series):
            if p.is_dir():
                return p

    return None

def resolve_selected(slots, label):
    records=[]
    unresolved=[]
    for study, rows in tqdm(slots.items(), desc=f"resolve {label}"):
        for slot, r in enumerate(rows):
            p=resolve_series_dir(study, r["series"])
            if p is not None:
                records.append({
                    "study":str(study),
                    "series":str(r["series"]),
                    "slot":slot,
                    "path":str(p),
                    "plane":r.get("plane",""),
                    "fluid":r.get("fluid",""),
                    "fat":r.get("fat",""),
                })
            else:
                unresolved.append((str(study),str(r["series"])))

    # If direct layout lookup failed, diagnose before doing any expensive
    # recursive scan. We inspect a tiny sample of the expected study paths.
    if not records and unresolved:
        log("Direct DICOM layout lookup found zero series.")
        log("First unresolved study/series:")
        for x in unresolved[:5]:
            log(f"  {x}")

        # A very targeted fallback: search only for the first study UID in
        # each candidate root. This is substantially safer than scanning
        # the whole 500 GB corpus.
        sample_study, sample_series = unresolved[0]
        for root in IMAGE_ROOTS:
            study_path = root/sample_study
            if study_path.is_dir():
                log(f"Found study directory at {study_path}")
                try:
                    children=list(study_path.iterdir())
                    log("Study children:")
                    for q in children[:20]:
                        log(f"  {q}")
                except Exception as e:
                    log(f"Could not inspect {study_path}: {e}")

    return pd.DataFrame(records)

train_selected=resolve_selected(train_slots,"train")
test_selected=resolve_selected(test_slots,"test")

log(f"resolved train series={len(train_selected)}")
log(f"resolved test series={len(test_selected)}")

if train_selected.empty:
    raise RuntimeError(
        "CSV metadata was found, but no DICOM series directory was found. "
        "The diagnostic lines immediately above show the candidate image "
        "roots and unresolved UID layout. Do not spend GPU time yet."
    )

In [ ]:
# 6. MINIMAL DICOM PIXEL DECODER
import pydicom
from PIL import Image

def list_dicom_files(series_dir):
    p = Path(series_dir)
    try:
        fs = [q for q in p.iterdir() if q.is_file()]
    except Exception:
        return []
    return sorted(fs)

def choose_slices(files, n=2):
    if not files:
        return []
    if len(files) <= n:
        return files
    idx = np.linspace(0, len(files)-1, n).round().astype(int)
    return [files[int(i)] for i in idx]

def read_image(path):
    ds = pydicom.dcmread(str(path), force=True)
    a = ds.pixel_array.astype(np.float32)
    lo, hi = np.percentile(a, [1, 99])
    if hi <= lo:
        hi = lo + 1.0
    a = np.clip((a-lo)/(hi-lo), 0, 1)
    a = (a*255).astype(np.uint8)
    return np.stack([a,a,a], axis=-1)

def prepare_image(a):
    return np.asarray(
        Image.fromarray(a).resize((IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.BILINEAR)
    )

## 7. Load local MedSigLIP

This cell uses only the attached local model. It does not call `from_pretrained()` without `local_files_only=True`, and it does not install packages from the internet.

The preferred route is a Transformers-compatible local MedSigLIP checkpoint. If the attached artifact is KerasHub-only, use a KerasHub-specific adapter rather than pretending the two APIs are interchangeable.

In [ ]:
# 7. LOCAL MEDSIGLIP LOAD
if MEDSIGLIP_DIR is None:
    hits = []
    for d in shallow_dirs(INPUT_ROOT):
        if "medsiglip" in d.name.lower() or "med-siglip" in d.name.lower():
            hits.append(d)
    if hits:
        MEDSIGLIP_DIR = hits[0]

if MEDSIGLIP_DIR is None:
    raise FileNotFoundError("Set MEDSIGLIP_DIR to the mounted local MedSigLIP model.")

from transformers import AutoModel, AutoProcessor

processor = AutoProcessor.from_pretrained(
    str(MEDSIGLIP_DIR),
    local_files_only=True
)

# One replica per GPU. Loading independently is intentional for simple dual-T4
# embedding extraction rather than heavyweight distributed orchestration.
replicas = []
for i, device in enumerate(DEVICES):
    model = AutoModel.from_pretrained(
        str(MEDSIGLIP_DIR),
        local_files_only=True,
        torch_dtype=torch.float16 if device.type == "cuda" else torch.float32,
    )
    model.eval().to(device)
    replicas.append(model)
    log(f"MedSigLIP replica {i} loaded on {device}")

# Release a temporary CUDA cache if possible.
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
# 8. STREAMING MEDSIGLIP EMBEDDINGS
def image_features(images, gpu_idx):
    device = DEVICES[gpu_idx]
    model = replicas[gpu_idx]
    batch = processor(images=images, return_tensors="pt")
    batch = {k: v.to(device) for k,v in batch.items()}

    with torch.inference_mode():
        if device.type == "cuda":
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                if hasattr(model, "get_image_features"):
                    z = model.get_image_features(**batch)
                else:
                    out = model(**batch)
                    z = getattr(out, "image_embeds", None)
                    if z is None:
                        z = getattr(out, "pooler_output", None)
                        if z is None:
                            z = out.last_hidden_state[:,0]
        else:
            if hasattr(model, "get_image_features"):
                z = model.get_image_features(**batch)
            else:
                out = model(**batch)
                z = getattr(out, "image_embeds", None)
                if z is None:
                    z = getattr(out, "pooler_output", None)
                    if z is None:
                        z = out.last_hidden_state[:,0]

    return F.normalize(z.float(), dim=-1).cpu().numpy().astype(np.float16)

def extract_selected(selected, label):
    meta = []
    chunks = []
    imgs = []
    imgmeta = []
    gpu = 0

    for r in tqdm(selected.itertuples(index=False), total=len(selected), desc=f"{label}"):
        files = choose_slices(list_dicom_files(r.path), SLICES_PER_SLOT)
        for si, f in enumerate(files):
            try:
                imgs.append(prepare_image(read_image(f)))
                imgmeta.append((str(r.study), int(r.slot), si))
            except Exception:
                continue

            if len(imgs) >= EMBED_BATCH:
                z = image_features(imgs, gpu)
                chunks.append(z)
                meta.extend(imgmeta)
                imgs.clear()
                imgmeta.clear()
                gpu = (gpu + 1) % len(DEVICES)

    if imgs:
        z = image_features(imgs, gpu)
        chunks.append(z)
        meta.extend(imgmeta)

    if not chunks:
        raise RuntimeError(f"No embeddings extracted for {label}")

    E = np.concatenate(chunks, axis=0)
    M = pd.DataFrame(meta, columns=["study","slot","slice"])
    log(f"{label}: embedding matrix={E.shape}, dtype={E.dtype}")
    return M, E

train_meta, train_E = extract_selected(train_selected, "train")
np.save(WORK_ROOT/"train_embeddings.fp16.npy", train_E)
train_meta.to_csv(WORK_ROOT/"train_embedding_meta.csv", index=False)

del train_E
gc.collect()
for d in DEVICES:
    if d.type == "cuda":
        torch.cuda.empty_cache()

In [ ]:
# 9. STUDY POOLING
train_E = np.load(WORK_ROOT/"train_embeddings.fp16.npy", mmap_mode="r")
train_meta = pd.read_csv(WORK_ROOT/"train_embedding_meta.csv")

def pool(meta, E):
    ids = meta["study"].astype(str).values
    unique = pd.unique(ids)
    out = []
    for sid in tqdm(unique, desc="pool studies"):
        ix = np.flatnonzero(ids == sid)
        z = np.asarray(E[ix], dtype=np.float32)
        out.append(np.concatenate([z.mean(0), z.max(0)]))
    return unique.astype(str), np.asarray(out, dtype=np.float32)

study_ids, X = pool(train_meta, train_E)
log(f"study features={X.shape}")

In [ ]:
# 10. TARGET MATRIX WITH MISSING-LABEL MASKING
train2 = train.copy()
train2[ID_COL] = train2[ID_COL].astype(str)
indexed = train2.set_index(ID_COL)

Y = np.full((len(study_ids), len(TARGETS)), np.nan, dtype=np.float32)
for j,t in enumerate(TARGETS):
    if t in target_map:
        vals = pd.to_numeric(indexed.reindex(study_ids)[target_map[t]], errors="coerce").values
        Y[:,j] = vals.astype(np.float32)

log("label coverage=" + np.array2string(np.nanmean(np.isfinite(Y), axis=0), precision=3))

In [ ]:
# 11. SMALL MULTI-LABEL RANKING HEAD
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

class KneeHead(nn.Module):
    def __init__(self, dim, hidden=512, n_targets=12):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, hidden),
            nn.GELU(),
            nn.Dropout(0.15),
            nn.Linear(hidden, n_targets),
        )
    def forward(self,x):
        return self.net(x)

def masked_bce(logits,y):
    m = torch.isfinite(y)
    return F.binary_cross_entropy_with_logits(logits[m], y[m]) if m.any() else logits.sum()*0

def pairwise_auc_loss(logits,y,max_pairs=4096):
    ls=[]
    for j in range(y.shape[1]):
        m=torch.isfinite(y[:,j])
        if m.sum()<4: continue
        s=logits[m,j]; t=y[m,j]
        p=s[t>0.5]; n=s[t<=0.5]
        if len(p)==0 or len(n)==0: continue
        if len(p)*len(n)>max_pairs:
            pi=torch.randint(len(p),(max_pairs,),device=s.device)
            ni=torch.randint(len(n),(max_pairs,),device=s.device)
            d=p[pi]-n[ni]
        else:
            d=(p[:,None]-n[None,:]).reshape(-1)
        ls.append(F.softplus(-d).mean())
    return torch.stack(ls).mean() if ls else logits.sum()*0

def macro_auc(y,s):
    vals=[]
    for j in range(y.shape[1]):
        m=np.isfinite(y[:,j])
        if m.sum()<2 or np.unique(y[m,j]).size<2: continue
        vals.append(roc_auc_score(y[m,j],s[m,j]))
    return float(np.mean(vals)) if vals else np.nan

In [ ]:
# 12. 4-FOLD HEAD TRAINING
scaler = StandardScaler()
X32 = scaler.fit_transform(X.astype(np.float32)).astype(np.float32)

kf = KFold(n_splits=4, shuffle=True, random_state=SEED)
oof = np.full_like(Y, np.nan, dtype=np.float32)
fold_models=[]

for fold,(tr,va) in enumerate(kf.split(X32)):
    if hours_left() < 0.45:
        log("runtime guard: stopping additional folds")
        break

    model=KneeHead(X32.shape[1],512,len(TARGETS)).to(DEVICES[0])
    opt=torch.optim.AdamW(model.parameters(),lr=2e-3,weight_decay=1e-4)

    xb=torch.from_numpy(X32[tr]).to(DEVICES[0])
    yb=torch.from_numpy(Y[tr]).to(DEVICES[0])
    xv=torch.from_numpy(X32[va]).to(DEVICES[0])

    best=-np.inf
    best_state=None

    for epoch in range(25):
        model.train()
        opt.zero_grad(set_to_none=True)
        logits=model(xb)
        loss=masked_bce(logits,yb)+0.25*pairwise_auc_loss(logits,yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
        opt.step()

        model.eval()
        with torch.inference_mode():
            pv=model(xv).float().cpu().numpy()
        auc=macro_auc(Y[va],pv)

        if np.isfinite(auc) and auc>best:
            best=auc
            best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}

        if epoch>=8 and np.isfinite(auc) and auc < best-0.01:
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    model.eval()
    with torch.inference_mode():
        pv=model(xv).float().cpu().numpy()
    oof[va]=pv
    fold_models.append(model.cpu())
    log(f"fold {fold}: AUC={macro_auc(Y[va],pv):.5f}")

    del xb,yb,xv,model
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

log(f"OOF image macro-AUC={macro_auc(Y,oof):.5f}")

## 13. Optional report branch

Gemma 4 4B is intentionally **not loaded here**. If a separate offline notebook has already produced report logits, this cell blends them conservatively.

This keeps the final image pipeline fast and makes the text contribution measurable.

In [ ]:
# 13. OPTIONAL PRECOMPUTED TEXT FUSION
from scipy.stats import rankdata

def rank01(x):
    x=np.asarray(x,float)
    out=np.full_like(x,np.nan)
    m=np.isfinite(x)
    if m.any():
        r=rankdata(x[m],method="average")
        out[m]=(r-1)/max(1,len(r)-1)
    return out

final_oof=oof.copy()

if TEXT_TRAIN.exists():
    txt=pd.read_csv(TEXT_TRAIN)
    tid=find_col(txt,["StudyInstanceUID","study_id","id"])
    if tid is not None:
        txt[tid]=txt[tid].astype(str)
        tx=txt.set_index(tid)
        for j,t in enumerate(TARGETS):
            if t not in tx.columns: continue
            s=pd.to_numeric(tx.reindex(study_ids)[t],errors="coerce").values
            a=rank01(final_oof[:,j]); b=rank01(s)
            m=np.isfinite(a)&np.isfinite(b)
            final_oof[m,j]=0.80*a[m]+0.20*b[m]
        log(f"OOF after text fusion={macro_auc(Y,final_oof):.5f}")
else:
    log("No precomputed text scores; image-only.")

In [ ]:
# 14. TEST EMBEDDINGS
if hours_left()<0.20:
    raise RuntimeError("Runtime budget is too low for test inference.")

test_meta,test_E=extract_selected(test_selected,"test")
np.save(WORK_ROOT/"test_embeddings.fp16.npy",test_E)
test_meta.to_csv(WORK_ROOT/"test_embedding_meta.csv",index=False)

del test_E
gc.collect()
for d in DEVICES:
    if d.type=="cuda": torch.cuda.empty_cache()

test_E=np.load(WORK_ROOT/"test_embeddings.fp16.npy",mmap_mode="r")
test_meta=pd.read_csv(WORK_ROOT/"test_embedding_meta.csv")
test_ids,Xt=pool(test_meta,test_E)
Xt=scaler.transform(Xt.astype(np.float32)).astype(np.float32)

xt=torch.from_numpy(Xt)
preds=[]
for m in fold_models:
    m.eval()
    with torch.inference_mode():
        preds.append(m(xt).numpy())
test_pred=np.mean(preds,axis=0)
log(f"test predictions={test_pred.shape}")

In [ ]:
# 15. OPTIONAL TEST TEXT FUSION
if TEXT_TEST.exists():
    txt=pd.read_csv(TEXT_TEST)
    tid=find_col(txt,["StudyInstanceUID","study_id","id"])
    if tid is not None:
        txt[tid]=txt[tid].astype(str)
        tx=txt.set_index(tid)
        for j,t in enumerate(TARGETS):
            if t not in tx.columns: continue
            s=pd.to_numeric(tx.reindex(test_ids)[t],errors="coerce").values
            a=rank01(test_pred[:,j]); b=rank01(s)
            m=np.isfinite(a)&np.isfinite(b)
            test_pred[m,j]=0.80*a[m]+0.20*b[m]
        log("Applied test text fusion.")

In [ ]:
# 16. SUBMISSION
submission=pd.DataFrame({"StudyInstanceUID":test_ids})
for j,t in enumerate(TARGETS):
    submission[t]=test_pred[:,j].astype(float)

if SAMPLE_SUB.exists():
    ss=pd.read_csv(SAMPLE_SUB, compression="infer")
    sid=find_col(ss,["StudyInstanceUID","study_id","id"])
    if sid is not None:
        base=ss[[sid]].copy()
        base.columns=["StudyInstanceUID"]
        pred_df=pd.DataFrame({"StudyInstanceUID":test_ids})
        for j,t in enumerate(TARGETS):
            pred_df[t]=test_pred[:,j]
        submission=base.merge(pred_df,on="StudyInstanceUID",how="left")

submission_path=WORK_ROOT/"submission.csv"
submission.to_csv(submission_path,index=False)

log(f"submission written: {submission_path}")
log(f"submission shape: {submission.shape}")
display(submission.head())

## Final checks

Before submitting:
1. Confirm the local MedSigLIP attachment loads without internet.
2. Confirm `train_selected` and `test_selected` are non-empty.
3. Confirm the embedding extraction actually alternates between `cuda:0` and `cuda:1`.
4. Check OOF macro-AUC before enabling text fusion.
5. Only use the text artifact if its OOF contribution improves macro-AUC.
6. Inspect `submission.csv` against the supplied sample submission.

The competition score is macro ROC-AUC over twelve abnormalities. A 99.9% accuracy guarantee is neither technically meaningful for this metric nor supported by the available competition evidence.